# 09 性能优化与 EXPLAIN

## 八股问答

### 1. 慢查询日志怎么配置？

```sql
-- 查看和开启（生产一般配置文件持久化，线上用长查询阈值 1 到 3 秒）
SHOW VARIABLES LIKE 'slow_query_log';
SHOW VARIABLES LIKE 'long_query_time';
SET GLOBAL slow_query_log = ON;
SET GLOBAL long_query_time = 1;
```

慢日志记录超过阈值的 SQL，配合 `mysqldumpslow` 或 `pt-query-digest` 聚合分析，
是定位问题的第一步。

### 2. EXPLAIN 各列怎么看？

| 列 | 含义 | 关注点 |
| --- | --- | --- |
| id | SELECT 查询块编号 | 同一查询通常为 1，子查询或 UNION 会分配不同编号 |
| type | 访问类型 | system：访问系统表；const：主键、唯一索引；从好到差：eq_ref > ref > range > index > ALL |
| possible_keys | 可能使用的索引 | 只看参考 |
| key | 实际使用的索引 | 为 NULL 说明没走索引 |
| key_len | 索引使用长度 | 判断联合索引用了几列 |
| Extra | 附加信息 | `Using index` 好；`Using filesort`、`Using temporary` 差 |

Extra 常见值：

- `Using index`：覆盖索引，不回表。
- `Using index condition`：索引下推，先按索引条件过滤再回表。
- `Using where`：存储引擎返回后 Server 层再过滤。
- `Using filesort`：无法用索引排序，需要额外排序，应优化。
- `Using temporary`：用了临时表，常见于 group by 无索引。
- `Using join buffer`：join 无索引可用，应给关联列加索引。

### 3. 如何分页比较高效？

`LIMIT 50000, 20` 慢在扫描 5 万行后丢弃。两种优化：

- 延迟关联：`SELECT * FROM t JOIN (SELECT id FROM t WHERE ... LIMIT 50000, 20)
  tmp ON t.id = tmp.id`，内层只走覆盖索引取主键。
- 游标分页：`WHERE id > 上一页最后一条 id ORDER BY id LIMIT 20`，每次只扫
  20 行，适合按主键翻页的场景。

### 4. count(*) 怎么优化？

InnoDB 没有计数器，`COUNT(*)` 必须扫描。优化：

- 走最小的二级索引，扫描页数最少（所以给低选择性但很小的列建索引也有用）。
- 高频计数用单独计数表，事务里和业务更新一起维护。
- 允许近似值时用 `SHOW TABLE STATUS` 的 rows 或 information_schema 统计。

### 5. join 怎么优化？

- 小表驱动大表：优化器一般自动选择，被驱动表关联列必须有索引。
- 关联列类型一致，避免隐式转换导致索引失效。
- `EXPLAIN` 中出现 `Using join buffer` 说明被驱动表没索引。
- 能拆成两次查询就不要强行 join 大结果集。

### 6. 整体优化步骤？

1. 慢查询日志找到 Top SQL。
2. EXPLAIN 看 type、key、rows、Extra。
3. 对症下药：加/改索引、改写 SQL、深分页改造。
4. 复测真实耗时，并用 `EXPLAIN ANALYZE` 看实际扫描行数。
5. 仍慢再看 Buffer Pool 命中率、脏页和刷盘（[07-BufferPool与内存](07-BufferPool与内存.md)）。

### 7. SQL 执行频次、慢查询日志、Profile 和 EXPLAIN 怎么配合？

| 工具 | 回答什么问题 | 重点看什么 |
| --- | --- | --- |
| SQL 执行频次 | 哪些 SQL 最常执行 | `performance_schema.events_statements_summary_by_digest` 或 `sys.statement_analysis` 的 `count_star`、`sum_timer_wait`、`avg_rows_examined` |
| 慢查询日志 | 哪些 SQL 单次执行慢 | `slow_query_log`、`long_query_time`，再用 `mysqldumpslow`、`pt-query-digest` 聚合 |
| Profile | 一条 SQL 各阶段耗了多少时间 | `Sending data`、`Sorting result`、`Creating tmp table`；`SHOW PROFILE` 已弃用，生产优先使用 Performance Schema |
| EXPLAIN | 优化器打算怎么执行 | `type`、`key`、`rows`、`filtered`、`Extra`；`EXPLAIN ANALYZE` 看实际耗时和扫描行数 |

配合顺序：

```text
SQL 执行频次 / 慢查询日志 → EXPLAIN → Profile / EXPLAIN ANALYZE → 优化 → 复测
```

例子：

```sql
-- ① 看整体：是不是 SELECT 太多了？
SHOW GLOBAL STATUS LIKE 'Com_select';   -- 查看 SELECT 执行总次数
SHOW GLOBAL STATUS LIKE 'Com_insert';   -- 查看 INSERT 执行总次数

-- ② 开启慢查询日志，找到慢 SQL
SET GLOBAL slow_query_log = ON;
SET GLOBAL long_query_time = 1;
-- 然后去慢查询日志里看到一条：SELECT * FROM orders WHERE user_id = 123; 花了 3.2 秒

-- ③ 用 profile 看这条 SQL 的时间都花在哪了
SET profiling = 1;
SELECT * FROM orders WHERE user_id = 123;
SHOW PROFILES;
SHOW PROFILE FOR QUERY 1;               -- 看各阶段耗时
-- 发现 "Sending data" 占了 3 秒 → 说明在大量扫描/读取数据

-- ④ 用 EXPLAIN 分析为什么扫描这么多数据
EXPLAIN SELECT * FROM orders WHERE user_id = 123;
-- 发现 type=ALL（全表扫描），key=NULL（没走索引）
-- → 结论：user_id 字段缺索引，加索引解决
```


## 面试话术

“我的排查顺序是：慢查询日志找 SQL，EXPLAIN 看 type 和 Extra，优先解决
filesort、temporary 和 ALL。深分页用游标或延迟关联；count 用计数表；
join 保证被驱动表关联列有索引。改完用 EXPLAIN ANALYZE 复测真实耗时，确认
不是靠感觉优化。如果 SQL 和索引都没问题，再往 Buffer Pool 命中率、刷盘和
机器资源方向排查。”


## 自查清单

- [ ] 能说出 type 从好到差的完整顺序
- [ ] 能解释 filesort、temporary、join buffer 三种 Extra
- [ ] 能在终端对比三种分页方案的执行计划
- [ ] 能解释 count(*) 为什么不快以及怎么优化
- [ ] 能按“慢日志 → EXPLAIN → 改写 → 复测”讲一遍自己的优化案例
